In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
import pandas as pd
import time
from datetime import datetime
import re

In [2]:
categories = {
    '정치': 'https://news.naver.com/section/100',
    '경제': 'https://news.naver.com/section/101',
    '사회': 'https://news.naver.com/section/102',
    '생활/문화': 'https://news.naver.com/section/103',
    'IT/과학': 'https://news.naver.com/section/105',
    '세계': 'https://news.naver.com/section/104'
}

NUM_ARTICLES_PER_CATEGORY = 10

In [7]:
service = Service(ChromeDriverManager().install())
options = webdriver.ChromeOptions()
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

driver = webdriver.Chrome(service=service, options=options)

In [8]:
def get_article_links(driver, category_url, num_articles):
    driver.get(category_url)
    time.sleep(3)

    article_links = []

    try:
        selectors = [
            'a.sa_text_lede',
            'a.sa_text_strong',
            '.sa_text a',
            '.cluster_text_headline a',
            '.cluster_text_lede a'
        ]

        for selector in selectors:
            elements = driver.find_elements(By.CSS_SELECTOR, selector)
            for element in elements:
                url = element.get_attribute('href')
                if (url and 'news.naver.com' in url and '/article/' in url
                    and '/comment/' not in url
                    and url not in article_links):
                    article_links.append(url)
                    if len(article_links) >= num_articles:
                        break
            if len(article_links) >= num_articles:
                break
        
        print(f'✓ {len(article_links)}개의 기사 링크 수집 완료')
    
    except Exception as e:
        print(f'✗ 기사 링크 수집 실패: {e}')
    
    return article_links[:num_articles]

In [9]:
def parse_article_detail(driver, article_url, category):
    driver.get(article_url)
    time.sleep(1.5)

    article_data = {
        'article_id': '',
        'title': '',
        'content': '',
        'url': article_url,
        'published_date': '',
        'source': '',
        'author': '',
        'category': category
    }

    try:
        match = re.search(r'article/(\d+)/(\d+)', article_url)
        if match:
            article_data['article_id'] = f'ART_{match.group(1)}_{match.group(2)}'
        else:
            article_data['article_id'] = f"ART_{datetime.now().strftime('%Y%m%d%H%M%S')}"
        
        title_selectors = [
            '#title_area span',
            '#ct .media_end_head_headline',
            '.media_end_head_headline',
            'h2#title_area',
            '.news_end_title'
        ]

        for selector in title_selectors:
            try:
                title_element = driver.find_element(By.CSS_SELECTOR, selector)
                if title_element.text.strip():
                    article_data['title'] = title_element.text.strip()
                    break
            except:
                continue
        
        content_selectors = [
            '#dic_area',
            'article#dic_area',
            '.go_trans._article_content',
            '._article_body_contents'
        ]

        for selector in content_selectors:
            try:
                content_element = driver.find_element(By.CSS_SELECTOR, selector)
                if content_element.text.strip():
                    article_data['content'] = content_element.text.strip()
                    break
            except:
                continue
        
        try:
            source_element = driver.find_element(By.CSS_SELECTOR, 'a.media_end_head_top_logo img')
            article_data['source'] = source_element.get_attribute('alt')
        except:
            try:
                source_element = driver.find_element(By.CSS_SELECTOR, '.media_end_head_top_logo_text')
                article_data['source'] = source_element.text.strip()
            except:
                pass
        
        try:
            date_element = driver.find_element(By.CSS_SELECTOR, 'span.media_end_head_info_datestamp_time, span[data-date-time]')
            date_text = date_element.get_attribute('data-date-time') or date_element.text
            article_data['published_date'] = date_text.strip()
        except:
            article_data['published_date'] = datetime.now().strftime('%Y-%m-%d %H:%M')
        
        try:
            author_element = driver.find_element(By.CSS_SELECTOR, 'em.media_end_head_journalist_name, span.byline_s')
            article_data['author'] = author_element.text.strip()
        except:
            pass
    
    except Exception as e:
        print(f' ✗ 파싱 오류: {e} ')
    
    return article_data

In [10]:
all_articles = []

for category_name, category_url in categories.items():
    print(f'\n{"="*60}')
    print(f"[{category_name}] 카테고리 수집 시작...")
    print(f"{'='*60}")

    article_links = get_article_links(driver, category_url, NUM_ARTICLES_PER_CATEGORY)

    for idx, article_url in enumerate(article_links, 1):
        print(f"  [{idx}/{len(article_links)}] {article_url}")
        article_data = parse_article_detail(driver, article_url, category_name)

        if article_data['title']:
            all_articles.append(article_data)
            print(f"  ✓ 수집 완료: {article_data['title'][:50]}...")
        else:
            print(f"  ✗ 수집 실패 - 제목을 찾을 수 없습니다.")
        
        time.sleep(0.5)


[정치] 카테고리 수집 시작...
✓ 10개의 기사 링크 수집 완료
  [1/10] https://n.news.naver.com/mnews/article/014/0005456806
  ✓ 수집 완료: '1억 수수 의혹' 강선우 "어떠한 돈도 받은 적 없다"...
  [2/10] https://n.news.naver.com/mnews/article/421/0008689788
  ✓ 수집 완료: 李대통령, 韓 대도약 이끌 5대 성장 전략 제시…"2026년은 대도약 원년"(종합)...
  [3/10] https://n.news.naver.com/mnews/article/052/0002294688
  ✓ 수집 완료: 청문회에서 '호통' 쿠팡 로저스...오늘도 '마이웨이'...
  [4/10] https://n.news.naver.com/mnews/article/366/0001133586
  ✓ 수집 완료: 文 만난 조국 “李정부 성공 위한 레드팀 역할 할 것”...
  [5/10] https://n.news.naver.com/mnews/article/001/0015823756
  ✓ 수집 완료: 李대통령, 현충원 참배…"대도약 원년 국민과 함께 열겠다"...
  [6/10] https://n.news.naver.com/mnews/article/008/0005299104
  ✓ 수집 완료: 김민석, 새해 인사 "내란 완전 청산, 민주주의 굳건히 하는 한 해 될 것"...
  [7/10] https://n.news.naver.com/mnews/article/055/0001320813
  ✓ 수집 완료: 김정은, 파병 군인 가족과 새해 공연…"평양·모스크바가 있다"...
  [8/10] https://n.news.naver.com/mnews/article/011/0004573653
  ✓ 수집 완료: [신년사]장동혁 "국민 삶 생각하면 '선거 승리' 따라올 것"...
  [9/10] https://n.news.naver.com/mnews/article/020/000368

In [11]:
df_articles = pd.DataFrame(all_articles)

In [12]:
df_articles

,article_id,title,content,url,published_date,source,author,category
0,ART_014_0005456806,"'1억 수수 의혹' 강선우 ""어떠한 돈도 받은 적 없다""",(출처=연합뉴스)\n\n[파이낸셜뉴스] 강선우 더불어민주당 의원이 2022년 지방선...,https://n.news.naver.com/mnews/article/014/000...,2026-01-01 10:25:11,파이낸셜뉴스,한승곤 기자,정치
1,ART_421_0008689788,"李대통령, 韓 대도약 이끌 5대 성장 전략 제시…""2026년은 대도약 원년""(종합)","[신년사] '지방·분배·안전·문화·평화' 5대 성장 천명\n""성장 패러다임 완전히 ...",https://n.news.naver.com/mnews/article/421/000...,2026-01-01 05:00:00,뉴스1,이기림 외 2명,정치
2,ART_052_0002294688,청문회에서 '호통' 쿠팡 로저스...오늘도 '마이웨이',[앵커]\n국회에서 이틀째 쿠팡 청문회가 이어지고 있습니다.\n\n어제 로저스 쿠팡...,https://n.news.naver.com/mnews/article/052/000...,2025-12-31 17:45:48,YTN,이승은 기자,정치
3,ART_366_0001133586,文 만난 조국 “李정부 성공 위한 레드팀 역할 할 것”,조국 조국혁신당 대표와 서왕진 원내대표를 비롯한 지도부가 1일 새해를 맞아 경남 양...,https://n.news.naver.com/mnews/article/366/000...,2026-01-01 15:12:07,조선비즈,이종현 기자,정치
4,ART_001_0015823756,"李대통령, 현충원 참배…""대도약 원년 국민과 함께 열겠다""","새해 일정 시작…참배 후 참모진·국무뮈원과 떡국 조찬\n\n\n이재명 대통령, 현충...",https://n.news.naver.com/mnews/article/001/001...,2026-01-01 08:45:53,연합뉴스,고동욱 기자,정치
5,ART_008_0005299104,"김민석, 새해 인사 ""내란 완전 청산, 민주주의 굳건히 하는 한 해 될 것""",[the300]\n김민석 국무총리가 지난해 12월 31일 오전 서울 종로구 정부서울...,https://n.news.naver.com/mnews/article/008/000...,2026-01-01 09:27:35,머니투데이,김지은 기자,정치
6,ART_055_0001320813,"김정은, 파병 군인 가족과 새해 공연…""평양·모스크바가 있다""",▲ 평양 5월1일 경기장에서 열린 신년 경축 행사에 참석한 김정은 북한 국무위원장\...,https://n.news.naver.com/mnews/article/055/000...,2026-01-01 09:26:22,SBS,김아영 기자,정치
7,ART_011_0004573653,"[신년사]장동혁 ""국민 삶 생각하면 '선거 승리' 따라올 것""","""더 낮은 자세로 국민 섬기는 정당 될 것""\n장동혁 국민의힘 대표가 새해 첫 날인...",https://n.news.naver.com/mnews/article/011/000...,2026-01-01 11:42:10,서울경제,이진석 기자,정치
8,ART_020_0003686138,봉하마을 참배한 정청래 “김대중 노무현 문재인의 꿈 계승”,정청래 더불어민주당 대표가 1일 오후 경남 김해 진영읍 봉하마을을 찾아 고 노무현 ...,https://n.news.naver.com/mnews/article/020/000...,2026-01-01 14:26:16,동아일보,송치훈 기자,정치
9,ART_015_0005231492,"이혜훈 이어 유승민도?…""총리 제안 왔지만 답변 안했다""",이재명 정부 '보수인사 기용설'에 직접 언급\n사진=연합뉴스\n\n이재명 정부 초대...,https://n.news.naver.com/mnews/article/015/000...,2026-01-01 10:57:12,한국경제,김봉구 기자,정치


In [13]:
output_filename = f"Articles_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"
df_articles.to_excel(output_filename, index=False, engine='openpyxl')

In [14]:
driver.quit()

In [15]:
df = pd.read_excel('Articles_20260101_164317.xlsx')
df.head()

,article_id,title,content,url,published_date,source,author,category
0,ART_014_0005456806,"'1억 수수 의혹' 강선우 ""어떠한 돈도 받은 적 없다""",(출처=연합뉴스)\n\n[파이낸셜뉴스] 강선우 더불어민주당 의원이 2022년 지방선...,https://n.news.naver.com/mnews/article/014/000...,2026-01-01 10:25:11,파이낸셜뉴스,한승곤 기자,정치
1,ART_421_0008689788,"李대통령, 韓 대도약 이끌 5대 성장 전략 제시…""2026년은 대도약 원년""(종합)","[신년사] '지방·분배·안전·문화·평화' 5대 성장 천명\n""성장 패러다임 완전히 ...",https://n.news.naver.com/mnews/article/421/000...,2026-01-01 05:00:00,뉴스1,이기림 외 2명,정치
2,ART_052_0002294688,청문회에서 '호통' 쿠팡 로저스...오늘도 '마이웨이',[앵커]\n국회에서 이틀째 쿠팡 청문회가 이어지고 있습니다.\n\n어제 로저스 쿠팡...,https://n.news.naver.com/mnews/article/052/000...,2025-12-31 17:45:48,YTN,이승은 기자,정치
3,ART_366_0001133586,文 만난 조국 “李정부 성공 위한 레드팀 역할 할 것”,조국 조국혁신당 대표와 서왕진 원내대표를 비롯한 지도부가 1일 새해를 맞아 경남 양...,https://n.news.naver.com/mnews/article/366/000...,2026-01-01 15:12:07,조선비즈,이종현 기자,정치
4,ART_001_0015823756,"李대통령, 현충원 참배…""대도약 원년 국민과 함께 열겠다""","새해 일정 시작…참배 후 참모진·국무뮈원과 떡국 조찬\n\n\n이재명 대통령, 현충...",https://n.news.naver.com/mnews/article/001/001...,2026-01-01 08:45:53,연합뉴스,고동욱 기자,정치


In [21]:
import neo4j
import os

import dotenv
dotenv.load_dotenv()

URI = os.environ.get('NEO4J_URI')
neo4j_user = os.environ.get('NEO4J_USERNAME')
neo4j_password = os.environ.get('NEO4J_PASSWORD')
AUTH = ('neo4j', os.getenv('NEO4J_PASSWORD', 'password'))

driver = neo4j.GraphDatabase.driver(URI, auth=AUTH)

In [22]:
def chunk_text(text, chunk_size=500, overlap=50):
    if pd.isna(text) or text == '':
        return []
    
    text = str(text)
    chunks = []

    for i in range(0, len(text), chunk_size - overlap):
        chunk = text[i:i + chunk_size]
        if chunk.strip():
            chunks.append(chunk.strip())
    
    return chunks

In [24]:
def clear_database(tx):
    tx.run('MATCH (n) DETACH DELETE n')

def create_constraints(tx):
    constraints = [
        'CREATE CONSTRAINT IF NOT EXISTS FOR (a: Article) REQUIRE a.article_id IS UNIQUE',
        'CREATE CONSTRAINT IF NOT EXISTS FOR (c: Content) REQUIRE c.content_id IS UNIQUE',
        'CREATE CONSTRAINT IF NOT EXISTS FOR (m: Media) REQUIRE m.name IS UNIQUE',
        'CREATE CONSTRAINT IF NOT EXISTS FOR (cat: Category) REQUIRE cat.name IS UNIQUE'
    ]

    for constraint in constraints:
        try:
            tx.run(constraint)
        except Exception as e:
            print(f"제약조건 생성 중 오류: {e}")

with driver.session() as session:
    session.execute_write(clear_database)
    session.execute_write(create_constraints)

In [25]:
def create_article_node(tx, article_data):
    query = '''
    MERGE (a: Article {article_id: $article_id})
    SET a.title = $title,
        a.url = $url,
        a.published_date = $published_date
    RETURN a
    '''
    tx.run(query,
           article_id=article_data['article_id'],
           title=article_data['title'],
           url=article_data['url'],
           published_date=article_data['published_date'])

def create_content_nodes(tx, article_id, content_chunks, article_data):
    for i, chunk in enumerate(content_chunks):
        content_id = f"{article_id}_chunk_{i}"

        query = '''
        MERGE (c:Content {content_id: $content_id})
        SET c.chunk = $chunk,
            c.article_id = $article_id,
            c.title = $title,
            c.url = $url,
            c.published_date = $published_date,
            c.chunk_index = $chunk_index
        '''
        tx.run(query,
               content_id=content_id,
               chunk=chunk,
               article_id=article_id,
               title=article_data['title'],
               url=article_data['url'],
               published_date=article_data['published_date'],
               chunk_index=i)
        
        relationship_query = '''
        MATCH (a:Article {article_id: $article_id})
        MATCH (c:Content {content_id: $content_id})
        MERGE (a)-[:HAS_CHUNK]->(c)
        '''
        tx.run(relationship_query,
               article_id=article_id,
               content_id=content_id)

def create_media_node_and_relationship(tx, article_id, source):
    if pd.isna(source) or source == '':
        return

    media_query = '''
    MERGE (m:Media {name: $source})
    RETURN m
    '''
    tx.run(media_query, source=source)

    relationship_query = '''
    MATCH (a:Article {article_id: $article_id})
    MATCH (m:Media {name: $source})
    MERGE (m)-[:PUBLISHED]->(a)
    '''
    tx.run(relationship_query,
           article_id=article_id,
           source=source)
    
def create_category_node_and_relationship(tx, article_id, category):
    if pd.isna(category) or category == '':
        return
    
    category_query = '''
    MERGE (cat:Category {name: $category})
    RETURN cat
    '''
    tx.run(category_query, category=category)

    relationship_query = '''
    MATCH (a:Article {article_id: $article_id})
    MATCH (cat:Category {name: $category})
    MERGE (a)-[:BELONGS_TO]->(cat)
    '''
    tx.run(relationship_query,
           article_id=article_id,
           category=category)

In [26]:
def build_graph_from_dataframe(df, chunk_size=500, overlap=50):
    with driver.session() as session:
        for idx, row in df.iterrows():
            try:
                article_id = row.get('article_id', '')

                article_data = {
                    'article_id': article_id,
                    'title': row.get('title', ''),
                    'url': row.get('url', ''),
                    'published_date': str(row.get('published_date', ''))
                }

                session.execute_write(create_article_node, article_data)

                if 'content' in row and pd.notna(row['content']) and row['content'] != '':
                    content_chunks = chunk_text(row['content'], chunk_size, overlap)
                    if content_chunks:
                        session.execute_write(create_content_nodes, article_id, content_chunks, article_data)
                
                if 'source' in row:
                    session.execute_write(create_media_node_and_relationship, article_id, row['source'])

                if 'category' in row:
                    session.execute_write(create_category_node_and_relationship, article_id, row['category'])
                
                if (idx + 1) % 10 == 0:
                    print(f"진행률: {idx + 1}/{len(df)} ({((idx + 1)/len(df)*100):.1f}%)")
            
            except Exception as e:
                print(f"기사 {idx} 처리 중 오류 발생: {e}")
                continue

In [27]:
with driver.session() as session:
    print('데이터베이스 초기화 중...')
    session.execute_write(clear_database)
    session.execute_write(create_constraints)

build_graph_from_dataframe(df, chunk_size=500, overlap=50)

데이터베이스 초기화 중...
진행률: 10/60 (16.7%)
진행률: 20/60 (33.3%)
진행률: 30/60 (50.0%)
진행률: 40/60 (66.7%)
진행률: 50/60 (83.3%)
진행률: 60/60 (100.0%)


In [2]:
import os
import neo4j
import dotenv
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.retrievers import VectorRetriever, VectorCypherRetriever, Text2CypherRetriever, ToolsRetriever
from neo4j_graphrag.embeddings.openai import OpenAIEmbeddings
from neo4j_graphrag.indexes import create_vector_index
from neo4j_graphrag.generation import RagTemplate, GraphRAG

dotenv.load_dotenv()

URI = os.getenv('NEO4J_URI')
AUTH = ('neo4j', os.getenv('NEO4J_PASSWORD'))
driver = neo4j.GraphDatabase.driver(URI, auth=AUTH)

llm = OpenAILLM(
    model_name='gpt-4o',
    model_params={'temperature': 0}
)
embedder = OpenAIEmbeddings(model='text-embedding-3-small')

In [3]:
with driver.session() as session:
    result = session.run('MATCH (c:Content) WHERE c.embedding IS NULL RETURN elementId(c) AS id, c.chunk AS text')
    records = result.data()

    for record in records:
        node_id = record['id']
        text = record['text']
        vector = embedder.embed_query(text)

        session.run('''
        MATCH (c) WHERE elementId(c) = $id
        SET c.embedding = $embedding
        ''', {'id': node_id, 'embedding': vector})

In [4]:
INDEX_NAME = 'content_vector_index'
DIMENSION = 1536

create_vector_index(
    driver,
    INDEX_NAME,
    label='Content',
    embedding_property='embedding',
    dimensions=DIMENSION,
    similarity_fn='cosine',
)

In [5]:
vector_retriever = VectorRetriever(
    driver=driver,
    index_name=INDEX_NAME,
    embedder = embedder
)

In [6]:
results = vector_retriever.search(query_text='트럼프 대통령 정책', top_k=3)

print(f"검색 결과 수: {len(results.items)}")
for i, item in enumerate(results.items):
    print(f"\n결과 {i+1}:")
    print(f"Content: {item.content[:200]}...")
    if item.metadata:
        print(f"Metadata: {item.metadata}")

검색 결과 수: 3

결과 1:
Content: {'embedding': None, 'article_id': 'ART_448_0000580677', 'chunk_index': 0, 'content_id': 'ART_448_0000580677_chunk_0', 'chunk': '도널드 트럼프 미국 대통령 /Reuters=연합뉴스\n도널드 트럼프 미국 대통령이 31일(현지시간) 대도시에 범죄 척결을 위해 배...
Metadata: {'score': 0.6993929147720337, 'nodeLabels': ['Content'], 'id': '4:099022cf-34ca-42f9-9923-7a6ccafea874:284'}

결과 2:
Content: {'embedding': None, 'article_id': 'ART_056_0012096715', 'chunk_index': 1, 'content_id': 'ART_056_0012096715_chunk_1', 'chunk': '제재 위험에 계속 직면해있다는 것을 추가로 시사한다"고 덧붙였습니다.\n\n스콧 베선트 재무장관은 "불법적인 마두로 정권이 미국에...
Metadata: {'score': 0.6933881044387817, 'nodeLabels': ['Content'], 'id': '4:099022cf-34ca-42f9-9923-7a6ccafea874:267'}

결과 3:
Content: {'embedding': None, 'article_id': 'ART_366_0001133511', 'chunk_index': 0, 'content_id': 'ART_366_0001133511_chunk_0', 'chunk': '미국 트럼프 행정부가 한국 국회가 통과시킨 정보통신망법 개정안에 대해 공식적으로 우려를 나타냈다.\n\n도널드 트럼프 미국 대통령...
Metadata: {'score': 0.6913937330245972, 'nodeLabels': ['Content'], 'id': '4:099022cf-34ca-4

c:\Users\Jo\AppData\Local\Programs\Python311\Lib\site-packages\neo4j_graphrag\retrievers\vector.py:201: DeprecationWarning: The default returned 'id' field in the search results will be removed. Please switch to using 'elementId' instead.
  search_query, search_params = get_search_query(


In [7]:
results = vector_retriever.search(query_text='북한', top_k=3)

print(f"검색 결과 수: {len(results.items)}")
for i, item in enumerate(results.items):
    print(f"\n결과 {i+1}:")
    print(f"Content: {item.content[:200]}...")
    if item.metadata:
        print(f"Metadata: {item.metadata}")

검색 결과 수: 3

결과 1:
Content: {'embedding': None, 'article_id': 'ART_030_0003385980', 'chunk_index': 0, 'content_id': 'ART_030_0003385980_chunk_0', 'chunk': "붉은 말의 해 병오년(丙午年), 거침없는 도약과 활력을 상징하는 기운 속에서 대한민국 반도체 산업이 새로운 출발선에 섰다. 경기도...
Metadata: {'score': 0.640516996383667, 'nodeLabels': ['Content'], 'id': '4:099022cf-34ca-42f9-9923-7a6ccafea874:236'}

결과 2:
Content: {'embedding': None, 'article_id': 'ART_025_0003493522', 'chunk_index': 6, 'content_id': 'ART_025_0003493522_chunk_6', 'chunk': '한국 등 주변국에선 군사력 증강을 위한 것이라며 우려한다.\nA : 안보 문서 개정이나 방위비 증액 등은 일본이 아시아에서 더 많...
Metadata: {'score': 0.6204962730407715, 'nodeLabels': ['Content'], 'id': '4:099022cf-34ca-42f9-9923-7a6ccafea874:275'}

결과 3:
Content: {'embedding': None, 'article_id': 'ART_032_0003418544', 'chunk_index': 1, 'content_id': 'ART_032_0003418544_chunk_1', 'chunk': '협력할 것”이라고 강조했다.\n\n앞서 중국 인민해방군 동부전구는 29~30일(현지시간) 육·해·공군과 로켓군 병력을 동원해 대만...
Metadata: {'score': 0.617620050907135, 'nodeLabels': ['Content'], 'id': '4:099022cf-34ca-42f

In [8]:
retrieval_query = '''
MATCH (content:Content)<-[:HAS_CHUNK]-(article:Article)
OPTIONAL MATCH (article)-[:BELONGS_TO]->(category:Category)<-[:BELONGS_TO]-(related_article:Article)
WHERE article <> related_article

RETURN
    content.content_id AS content_id,
    content.chunk AS chunk,
    content.title AS content_title,
    article.article_id AS article_id,
    article.title AS article_title,
    article.url AS article_url,
    article.published_date AS article_date,
    category.name AS category_name,
    collect(DISTINCT {
        article_id: related_article.article_id,
        title: related_article.title,
        url: related_article.url,
        published_date: related_article.published_date
    }) AS related_articles

ORDER BY article.published_date DESC
LIMIT 1
'''

In [9]:
vector_cypher_retriever = VectorCypherRetriever(
    driver=driver,
    index_name=INDEX_NAME,
    retrieval_query=retrieval_query,
    embedder=embedder
)

In [10]:
results = vector_cypher_retriever.search(query_text='삼성전자')

print(f"검색 결과 수: {len(results.items)}")
for i, item in enumerate(results.items):
    print(f"\n결과 {i+1}:")
    print(f"Content: {item.content[:200]}...")
    print(item.content)

c:\Users\Jo\AppData\Local\Programs\Python311\Lib\site-packages\neo4j_graphrag\retrievers\vector.py:368: DeprecationWarning: The default returned 'id' field in the search results will be removed. Please switch to using 'elementId' instead.
  search_query, search_params = get_search_query(


검색 결과 수: 1

결과 1:
Content: <Record content_id='ART_015_0005231558_chunk_0' chunk="미국 증시 전망\n\nAI 거품론 vs 대세론 시험대\n기업 영업이익률 고점 경신 등\n'상승 랠리' 올해도 이어질 듯\n\n데이터센터·전력 수요 폭증에\n에너지인프라·유틸리티株 유망\n\n뜨거웠던 비만약 열풍 이어져\n헬스케어·바이오 등 혁신 기업\n'제2 ...
<Record content_id='ART_015_0005231558_chunk_0' chunk="미국 증시 전망\n\nAI 거품론 vs 대세론 시험대\n기업 영업이익률 고점 경신 등\n'상승 랠리' 올해도 이어질 듯\n\n데이터센터·전력 수요 폭증에\n에너지인프라·유틸리티株 유망\n\n뜨거웠던 비만약 열풍 이어져\n헬스케어·바이오 등 혁신 기업\n'제2 엔비디아' 될 가능성 커\n올해 미국 증시는 지난 2년간 기술주 열풍을 이끈 빅테크에 냉혹한 시험대가 될 것으로 전망된다. 이들 기업이 인공지능(AI) 부문에서 가시적 이익을 낼 수 있어야 주가 거품 논란에서 벗어나기 때문이다. 월가에선 작년보다 올해 미국의 기준금리 인하 속도가 둔화해 부채가 적고 현금 흐름이 좋은 기업에 투자자 시선이 쏠릴 것이란 예상이 나온다.\n\nAI 수익화의 원년\n골드만삭스는 최근 보고서를 통해 2026년 S&P500지수가 7000 선에 안착할 것이라는 낙관론을 내놨다. 그 근거는 AI 인프라 투자 결과물이 기업 주당순이익(EPS)으로 본격 전환되는 ‘수확의 시기’가 도래한다는 점이다.\n\n사비타 수브라마니안 뱅크오브아메리카(B" content_title='"S&P500, 7000선 안착…돈되는 AI株 담아야"' article_id='ART_015_0005231558' article_title='"S&P500, 7000선 안착…돈되는 AI株 담아야"' article_url='https://n.news.naver.com/mnews/article/015/0005231558' a

In [11]:
import re
import json
import ast

def extract_field(text, name):
    pattern = rf"{name}\s*=\s*(?P<val>'[^']*'|\"[^\"]*\"|'''(?:.|\n)*?'''|\"\"\"(?:.|\n)*?\"\"\")"
    m = re.search(pattern, text)
    
    val = m.group("val")

    return val[1:-1]
    
def extract_related(text):
    m = re.search(r"related_articles\s*=\s*(\[.*?\])", text, re.DOTALL)
    raw = m.group(1)

    return ast.literal_eval(raw)

def parse_content(text):
    fields = [
        'content_id', 'chunk', 'content_title',
        'article_id', 'article_title', 'article_url',
        'article_date', 'category_name'
    ]

    extracted = {f: extract_field(text, f) for f in fields}
    extracted['related_articles'] = extract_related(text)

    if extracted['chunk']:
        extracted['chunk'] = extracted['chunk'].replace('\\n', '\n')

    return extracted

result = parse_content(str(item.content))

print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "content_id": "ART_015_0005231558_chunk_0",
  "chunk": "미국 증시 전망\n\nAI 거품론 vs 대세론 시험대\n기업 영업이익률 고점 경신 등\n'상승 랠리' 올해도 이어질 듯\n\n데이터센터·전력 수요 폭증에\n에너지인프라·유틸리티株 유망\n\n뜨거웠던 비만약 열풍 이어져\n헬스케어·바이오 등 혁신 기업\n'제2 엔비디아' 될 가능성 커\n올해 미국 증시는 지난 2년간 기술주 열풍을 이끈 빅테크에 냉혹한 시험대가 될 것으로 전망된다. 이들 기업이 인공지능(AI) 부문에서 가시적 이익을 낼 수 있어야 주가 거품 논란에서 벗어나기 때문이다. 월가에선 작년보다 올해 미국의 기준금리 인하 속도가 둔화해 부채가 적고 현금 흐름이 좋은 기업에 투자자 시선이 쏠릴 것이란 예상이 나온다.\n\nAI 수익화의 원년\n골드만삭스는 최근 보고서를 통해 2026년 S&P500지수가 7000 선에 안착할 것이라는 낙관론을 내놨다. 그 근거는 AI 인프라 투자 결과물이 기업 주당순이익(EPS)으로 본격 전환되는 ‘수확의 시기’가 도래한다는 점이다.\n\n사비타 수브라마니안 뱅크오브아메리카(B",
  "content_title": "\"S&P500, 7000선 안착…돈되는 AI株 담아야\"",
  "article_id": "ART_015_0005231558",
  "article_title": "\"S&P500, 7000선 안착…돈되는 AI株 담아야\"",
  "article_url": "https://n.news.naver.com/mnews/article/015/0005231558",
  "article_date": "2026-01-01 16:13:14",
  "category_name": "세계",
  "related_articles": [
    {
      "article_id": "ART_023_0003950359",
      "title": "새해 최고 부자는 머스크 895조원... 테크 창업자가 상위권 휩쓸어",
   

In [12]:
def get_schema():
    with driver.session() as session:
        node_info = session.run("""
            CALL db.schema.nodeTypeProperties()
            YIELD nodeType, propertyName, propertyTypes
            RETURN nodeType, collect(propertyName) as properties
        """).data()

        rel_info = session.run('''
            CALL db.schema.relTypeProperties()
            YIELD relType, propertyName, propertyTypes
            RETURN relType, collect(propertyName) as properties
        ''').data()

        patterns = session.run('''
            MATCH (n)-[r]->(m)
            RETURN DISTINCT labels(n)[0] as source, type(r) as relationship, labels(m)[0] as target
            LIMIT 20
        ''').data()

        schema_text = '=== Neo4j Schema ===\n'

        schema_text += '\n노드 타입:\n'
        for node in node_info:
            schema_text += f"- {node['nodeType']}: {node['properties']}\n"

        schema_text += '\n관계 패턴:\n'
        for pattern in patterns:
            schema_text += f"- ({pattern['source']})-[:{pattern['relationship']}]->({pattern['target']})\n"
        
        return schema_text

neo4j_schema = get_schema()
print(neo4j_schema)

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Procedure.ProcedureWarning} {category: GENERIC} {title: The query used a procedure that generated a warning.} {description: The query used a procedure that generated a warning. (The field `propertyTypes` will change output format in the next major version.)} {position: line: 2, column: 13, offset: 13} for query: '\n            CALL db.schema.nodeTypeProperties()\n            YIELD nodeType, propertyName, propertyTypes\n            RETURN nodeType, collect(propertyName) as properties\n        '
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Procedure.ProcedureWarning} {category: GENERIC} {title: The query used a procedure that generated a warning.} {description: The query used a procedure that generated a warning. (The field `propertyTypes` will change output format in the next major version.)} {position: line: 2, column: 13, offset: 13} for query: '\n    

=== Neo4j Schema ===

노드 타입:
- :`Article`: ['article_id', 'title', 'url', 'published_date']
- :`Content`: ['article_id', 'content_id', 'title', 'url', 'published_date', 'chunk', 'chunk_index', 'embedding']
- :`Media`: ['name']
- :`Category`: ['name']

관계 패턴:
- (Article)-[:HAS_CHUNK]->(Content)
- (Article)-[:BELONGS_TO]->(Category)
- (Media)-[:PUBLISHED]->(Article)



In [13]:
examples = [
    '''
    USER INPUT: 경제 분야의 최신 뉴스 알려주세요
    CYPHER QUERY:
    MATCH (a:Article)-[:BELONGS_TO]->(c:Category {name: "경제"})
    RETRN a.article_id, a.title, a.url, a.published_date
    ORDER BY a.published_date DESC
    LIMIT 10
    ''',
    '''
    USER INPUT: 매일경제에서 나온 최신 뉴스 3개 보여주세요
    CYPHER QUERY:
    MATCH (m:Media {name: "매일경제"})-[:PUBLISHED]->(a:Article)
    RETURN a.article_id, a.title, a.url, a.published_date
    ORDER BY a.published_date DESC
    LIMIT 3
    ''',
    '''
    USER INPUT: 2025년 11월 1일 이후에 발행된 정치 관련 기사는 몇 개나 되나요?
    CYPHER QUERY:
    MATCH (a:Article)-[:BELONGS_TO]->(c:Category)
    RETURN c.name as category, count(a) as article_count
    ORDER BY article_count DESC
    ''',
    '''
    USER INPUT: 11월 2일에 발행된 기사 중 정치 분야는?
    CYPHER QUERY:
    MATCH (a:Article)-[:BELONGS_TO]->(c:Category {name: "정치"})
    WHERE a.published_date STARTS WITH "2025-11-02"
    RETURN a.article_id, a.title, a.url, a.published_date
    ORDER BY a.published_date DESC
    ''',
]

text2cypher_retriever = Text2CypherRetriever(
    driver=driver,
    llm=llm,
    neo4j_schema=neo4j_schema,
    examples=examples,
)

In [14]:
results = text2cypher_retriever.search(query_text='정치 카테고리의 최신 기사 5개를 보여주세요')
for j, item in enumerate(results.items):
    print(f" 결과 {j+1}:")
    print(f' Content: {item.content}')
    if item.metadata:
        print(f" Metadata: {item.metadata}")

 결과 1:
 Content: <Record a.article_id='ART_366_0001133586' a.title='文 만난 조국 “李정부 성공 위한 레드팀 역할 할 것”' a.url='https://n.news.naver.com/mnews/article/366/0001133586' a.published_date='2026-01-01 15:12:07'>
 결과 2:
 Content: <Record a.article_id='ART_020_0003686138' a.title='봉하마을 참배한 정청래 “김대중 노무현 문재인의 꿈 계승”' a.url='https://n.news.naver.com/mnews/article/020/0003686138' a.published_date='2026-01-01 14:26:16'>
 결과 3:
 Content: <Record a.article_id='ART_011_0004573653' a.title='[신년사]장동혁 "국민 삶 생각하면 \'선거 승리\' 따라올 것"' a.url='https://n.news.naver.com/mnews/article/011/0004573653' a.published_date='2026-01-01 11:42:10'>
 결과 4:
 Content: <Record a.article_id='ART_015_0005231492' a.title='이혜훈 이어 유승민도?…"총리 제안 왔지만 답변 안했다"' a.url='https://n.news.naver.com/mnews/article/015/0005231492' a.published_date='2026-01-01 10:57:12'>
 결과 5:
 Content: <Record a.article_id='ART_014_0005456806' a.title='\'1억 수수 의혹\' 강선우 "어떠한 돈도 받은 적 없다"' a.url='https://n.news.naver.com/mnews/article/014/0005456806' a.published_date='

In [15]:
results = text2cypher_retriever.search(query_text='언론사별로 기사 개수를 알려주세요')
for j, item in enumerate(results.items):
    print(f" 결과 {j+1}:")
    print(f" Content: {item.content}")
    if item.metadata:
        print(f" Metadata: {item.metadata}")

 결과 1:
 Content: <Record media_name='연합뉴스' article_count=7>
 결과 2:
 Content: <Record media_name='머니투데이' article_count=6>
 결과 3:
 Content: <Record media_name='YTN' article_count=4>
 결과 4:
 Content: <Record media_name='한국경제' article_count=3>
 결과 5:
 Content: <Record media_name='매일경제' article_count=3>
 결과 6:
 Content: <Record media_name='한국일보' article_count=3>
 결과 7:
 Content: <Record media_name='파이낸셜뉴스' article_count=2>
 결과 8:
 Content: <Record media_name='뉴스1' article_count=2>
 결과 9:
 Content: <Record media_name='조선비즈' article_count=2>
 결과 10:
 Content: <Record media_name='서울경제' article_count=2>
 결과 11:
 Content: <Record media_name='동아일보' article_count=2>
 결과 12:
 Content: <Record media_name='조선일보' article_count=2>
 결과 13:
 Content: <Record media_name='KBS' article_count=2>
 결과 14:
 Content: <Record media_name='전자신문' article_count=2>
 결과 15:
 Content: <Record media_name='경향신문' article_count=2>
 결과 16:
 Content: <Record media_name='디지털타임스' article_count=2>
 결과 17:
 Content: <Record media